# Experiment arms eval (EXP-016 and later) - PRIVATE
Runs experiment **arms** (`experiments/arms/<id>/arm.yaml`) through the official `swegemma` harness on 4xL4, started
the way the scorer starts it, and writes a paired scorecard. Every arm is scored with its own `eval_config.yaml`.

* Default: `ctrl_022` (the 0.22 zip) vs `exp016_a0_pruned` vs `exp016_a1_skill_first` on **DEV-30**, 4 tasks in flight.
* Overrides (`scripts/kaggle_push.py -exp016_eval --set key=value`):
  `tasks` = `dev30` | `all` | `smoke5` | `"id1,id2"`; `arms` = `"a,b,c"`; `concurrency`; `clock_hours`; `mode` = `async` | `threads`; `control`.
* 5-task smoke first: `--set tasks=smoke5 --set arms=exp016_a1_skill_first`.
* Outputs under `/kaggle/working/exp_eval/<label>/`: per arm `submission.zip`, `arm_manifest.json`, `rows.jsonl`,
  `patches/`, `harness/` (traces + logs); plus `scorecard.md` / `scorecard.json` and `driver.log`.
  Traces contain task text: keep this notebook private.


In [ ]:
import json, os, shutil, subprocess, sys, time
from pathlib import Path

T_START = time.time()
RUN_OVERRIDES = {}  # @@RUN_OVERRIDES@@
RUN = {
    "arms": "ctrl_022,exp016_a0_pruned,exp016_a1_skill_first",
    "control": "ctrl_022",
    "tasks": "dev30",
    "concurrency": 4,
    "clock_hours": 11.5,     # the whole notebook must end before Kaggle's 12 h kill
    "mode": "async",         # "threads" if run_summary shows effective_parallelism ~1 (harness call blocks)
    "max_num_seqs": 32,
    "label": None,
    **RUN_OVERRIDES,
}
if isinstance(RUN["arms"], list):
    RUN["arms"] = ",".join(RUN["arms"])
if isinstance(RUN["tasks"], list):
    RUN["tasks"] = ",".join(RUN["tasks"])
RUN["label"] = RUN["label"] or f"{RUN['tasks'] if len(RUN['tasks']) < 20 else 'custom'}_{time.strftime('%Y%m%d_%H%M')}"
print(json.dumps(RUN, indent=1))
INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")

def walk(pred, root=INPUT, depth=7):
    hits = []
    for dp, dn, fn in os.walk(root, followlinks=True):
        d = Path(dp)
        if pred(d, set(fn)):
            hits.append(d); dn[:] = []; continue
        dn[:] = [] if len(d.relative_to(root).parts) >= depth else sorted(dn)
    return hits


In [ ]:
# 1) offline install, same recipe as the training notebook (official wheelhouse first, then ours, both --no-deps)
whs = walk(lambda d, f: any(n.endswith(".whl") for n in f), depth=4)
print("wheelhouses:", [str(w) for w in whs])
def install(wh, force):
    dst = Path("/tmp/wh") / wh.name; dst.mkdir(parents=True, exist_ok=True)
    files = []
    for w in sorted(wh.glob("*.whl")):
        n = w.name.replace("cu128", "+cu128", 1) if (w.name.startswith("torch-") and "cu128" in w.name and "+cu128" not in w.name) else w.name
        if not (dst / n).exists(): shutil.copy2(w, dst / n)
        files.append(str(dst / n))
    cmd = [sys.executable, "-m", "pip", "install", "-q", "--no-index", "--find-links", str(dst), "--no-deps"] + (["--force-reinstall"] if force else [])
    lock = wh / "requirements.lock.txt"
    items = [l.strip() for l in lock.read_text().splitlines() if l.strip() and not l.startswith("#")] if lock.is_file() else files
    r = subprocess.run(cmd + items); print(wh.name, "pip rc", r.returncode)
official = [w for w in whs if "developer-agent" in str(w).lower()]
for w in official: install(w, True)
for w in whs:
    if w not in official and 'tpu' not in w.name.lower(): install(w, False)  # skip the TPU wheelhouse: libtpu makes vLLM detect a tpu platform
# our wheelhouse (TPU training stack) drags in a vLLM TPU platform plugin -> "Only one platform plugin can be activated: cuda, tpu". Remove it.
from importlib import metadata as _md
_pk = ["libtpu", "libtpu-nightly", "tpu-info", "torch-xla", "torch_xla", "jax", "jaxlib", "tpu-inference", "vllm-tpu"]
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q"] + _pk)
_bad = {e.dist.name for e in _md.entry_points(group="vllm.platform_plugins") if e.name != "cuda" and e.dist is not None}
print("non-cuda vllm platform plugins:", _bad)
if _bad: subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q"] + sorted(_bad))
print("remaining platform plugins:", [(e.name, e.dist.name) for e in _md.entry_points(group="vllm.platform_plugins")])
r = subprocess.run([sys.executable, "-c", "import vllm, adk_submission, swegemma; print('vllm', vllm.__version__)"], capture_output=True, text=True)
print(r.stdout or r.stderr[-1500:])

In [ ]:
# 2) locate the repo dataset, the w4a16 base model and the competition data; check the arms exist
repo_dirs = walk(lambda d, f: (d / "pipeline" / "exp_eval" / "kaggle_run.py").is_file(), depth=5)
assert repo_dirs, "repo dataset has no pipeline/exp_eval: commit + push + update the banwait13/gemmafinetuning dataset"
REPO = repo_dirs[0]
BASE = walk(lambda d, f: "config.json" in f and "w4a16" in str(d).lower(), depth=8)[0]
COMP = walk(lambda d, f: "tasks.jsonl" in f and (d / "snapshots").is_dir(), depth=5)[0]
print("REPO:", REPO)
print("BASE:", BASE)
print("COMP:", COMP)
missing = [a for a in RUN["arms"].split(",") if not (REPO / "experiments" / "arms" / a / "arm.yaml").is_file()]
assert not missing, f"arms not in the repo dataset: {missing}"
OUT = WORK / "exp_eval" / RUN["label"]
OUT.mkdir(parents=True, exist_ok=True)


In [ ]:
# 3) the driver runs in its own process (a notebook kernel already has an event loop): build arms, start vLLM once,
#    run arms x tasks with RUN["concurrency"] in flight (arms interleaved task by task), then write the scorecard.
deadline_ts = T_START + float(RUN["clock_hours"]) * 3600 - 15 * 60  # 15 min left for the tally and output upload
cmd = [sys.executable, "-B", "-m", "pipeline.exp_eval.kaggle_run",
       "--base-model", str(BASE), "--comp", str(COMP), "--out", str(OUT),
       "--arms", RUN["arms"], "--tasks", str(RUN["tasks"]), "--control", RUN["control"],
       "--concurrency", str(RUN["concurrency"]), "--deadline-ts", str(deadline_ts),
       "--mode", RUN["mode"], "--max-num-seqs", str(RUN["max_num_seqs"])]
env = {**os.environ, "PYTHONDONTWRITEBYTECODE": "1", "PYTHONUNBUFFERED": "1"}
print(" ".join(cmd), flush=True)
with open(OUT / "driver.log", "w") as lf:
    p = subprocess.Popen(cmd, cwd=REPO, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        lf.write(line); lf.flush()
        if not line.startswith(("INFO", "DEBUG")) or "Error" in line:
            print(line, end="", flush=True)
    rc = p.wait()
print("driver exit code:", rc)


In [ ]:
# 4) scorecard + what was written
sc = OUT / "scorecard.md"
print(sc.read_text() if sc.is_file() else "NO SCORECARD: read driver.log above")
for arm_dir in sorted(p for p in OUT.iterdir() if p.is_dir()):
    s = arm_dir / "run_summary.json"
    print(arm_dir.name, json.loads(s.read_text()) if s.is_file() else "(no run_summary.json)")
shutil.make_archive(str(WORK / f"exp_eval_{RUN['label']}"), "zip", OUT)  # one file to download
print("elapsed hours:", round((time.time() - T_START) / 3600, 2))
